In [3]:
import csv
import os
from pathlib import Path

import pandas as pd

COMMON_VOICE_ROOTS = [
    Path("/kaggle/input/datasets/manofwar/commonvoice-ar-ru"),
    Path("/kaggle/input/datasets/manofwar/mozila-id-et-el-fi"),
]
MUSAN_ROOT = Path("/kaggle/input/datasets/dogrose/musan-dataset")
COLUMNS = ["locale", "speaker_id", "audio_path", "source", "duration"]
MUSAN_SOURCES = ("librivox", "music", "noise")  # из speech берём только librivox
LIBRIVOX_CODES = {
    "arabic": "ar", "chinese": "zh", "danish": "da", "dutch": "nl", "english": "en",
    "french": "fr", "german": "de", "hebrew": "he", "hungarian": "hu", "italian": "it",
    "japanese": "ja", "latin": "la", "polish": "pl", "portuguese": "pt", "russian": "ru",
    "spanish": "es", "tagalog": "tl",
}


def read_tsv(path, columns):
    return pd.read_csv(path, sep="\t", quoting=csv.QUOTE_NONE, dtype=str, keep_default_na=False,
                       usecols=columns, encoding="utf-8-sig")


def common_voice_samples(root, table="validated.tsv"):
    """Все записи из table каждого языка Common Voice внутри root."""
    frames = []
    for folder, subfolders, files in os.walk(root):
        subfolders[:] = sorted(d for d in subfolders if d != "clips")  # в clips сотни тысяч файлов
        if table not in files:
            continue
        lang_dir = Path(folder)
        clips = read_tsv(lang_dir / table, ["client_id", "path"])
        speaker = clips["client_id"].str.strip()
        frame = pd.DataFrame({
            "locale": lang_dir.name,
            "speaker_id": speaker.mask(speaker == ""),
            "audio_path": str(lang_dir / "clips") + os.sep + clips["path"],
            "source": "commonvoice",
        })
        if "clip_durations.tsv" in files:
            durations = read_tsv(lang_dir / "clip_durations.tsv", ["clip", "duration[ms]"])
            durations = durations.drop_duplicates("clip").set_index("clip")["duration[ms]"]
            frame["duration"] = clips["path"].map(pd.to_numeric(durations, errors="coerce") / 1000)
        frames.append(frame)
    return frames


def musan_samples(root):
    """WAV из MUSAN speech/librivox, music и noise; длительности в MUSAN нет."""
    rows = []
    for folder, subfolders, files in os.walk(root):
        subfolders.sort()
        parts = Path(folder).relative_to(root).parts
        source = next((name for name in MUSAN_SOURCES if name in parts), None)
        if source is None:
            continue
        languages = {}
        if source == "librivox" and "ANNOTATIONS" in files:
            for line in (Path(folder) / "ANNOTATIONS").read_text(encoding="utf-8-sig").splitlines():
                fields = line.split()
                if len(fields) >= 3 and not fields[0].startswith("#"):
                    languages[fields[0]] = LIBRIVOX_CODES.get(fields[2].lower(), fields[2].lower())
        for name in sorted(files):
            if name.endswith(".wav"):
                rows.append({"locale": languages.get(name[:-4]), "speaker_id": None,
                             "audio_path": os.path.join(folder, name), "source": f"{source}"})
    return pd.DataFrame(rows, columns=COLUMNS)


def collect_samples(cv_roots=COMMON_VOICE_ROOTS, musan_root=MUSAN_ROOT, table="validated.tsv"):
    """Одна таблица: locale, speaker_id, audio_path, source, duration (секунды или NaN)."""
    frames = [frame for root in cv_roots for frame in common_voice_samples(root, table)]
    frames.append(musan_samples(musan_root))
    samples = pd.concat(frames, ignore_index=True).reindex(columns=COLUMNS)
    samples["duration"] = samples["duration"].astype("float64")
    return samples


samples = collect_samples()
samples.groupby(["source", "locale"], dropna=False).size()

source       locale
commonvoice  ar         80696
             el         17653
             et         28375
             fi         12310
             id         30901
             ru        178296
librivox     ar             1
             da             1
             de            16
             en            76
             es            11
             fr            11
             he             4
             hu             1
             it             8
             ja             3
             la             1
             nl             9
             pl             2
             pt             9
             ru             9
             tl             3
             zh             7
             NaN            1
music        NaN          660
noise        NaN          930
dtype: int64

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd

TARGET_LOCALES = ["ru", "ar"]          # 90% дикторов в train, остальные поровну в dev и test
UNKNOWN_TRAIN_LOCALES = ["el", "id"]   # все дикторы в train
UNKNOWN_EVAL_LOCALES = ["et", "fi"]    # все дикторы поровну в dev и test
TRAIN_SHARE = 0.9                      # доля дикторов ru/ar в train
MUSAN_TRAIN_SHARE = 0.3                # доля записей MUSAN в train, отдельно для librivox, music и noise
MAX_CLIPS_PER_SPEAKER = {"train": 30, "dev": 5, "test": 5}  # None — без лимита
SEED = 42

rng = np.random.default_rng(SEED)


def assign(keys, train_share):
    """Доля train_share ключей — в train, остальные по очереди в dev и test."""
    keys = rng.permutation(np.array(sorted(keys), dtype=object))
    n_train = round(len(keys) * train_share)
    rest = keys[n_train:]
    first, second = ("dev", "test") if rng.random() < 0.5 else ("test", "dev")  # нечётный остаток — наугад
    return {**dict.fromkeys(keys[:n_train], "train"),
            **dict.fromkeys(rest[0::2], first), **dict.fromkeys(rest[1::2], second)}


cv = samples["source"] == "commonvoice"
speakers_of = lambda locale: set(samples.loc[cv & (samples["locale"] == locale), "speaker_id"].dropna())

# Диктор Common Voice попадает ровно в одну выборку, даже если записывался на нескольких языках.
speaker_split = {}
for locale in UNKNOWN_TRAIN_LOCALES:
    speaker_split.update(dict.fromkeys(speakers_of(locale), "train"))
for locale in TARGET_LOCALES:
    speaker_split.update(assign(speakers_of(locale) - speaker_split.keys(), TRAIN_SHARE))
for locale in UNKNOWN_EVAL_LOCALES:
    speaker_split.update(assign(speakers_of(locale) - speaker_split.keys(), 0.0))

split = pd.Series(np.nan, index=samples.index, dtype=object)
known = cv & samples["locale"].isin(TARGET_LOCALES + UNKNOWN_TRAIN_LOCALES + UNKNOWN_EVAL_LOCALES)
split[known] = samples.loc[known, "speaker_id"].map(speaker_split)
leaked = cv & samples["locale"].isin(UNKNOWN_EVAL_LOCALES) & (split == "train")
split[leaked] = np.nan  # et/fi от дикторов, которые уже в train через другой язык

# MUSAN делится отдельно внутри каждой группы: librivox — по языку, music и noise — по подпапке.
musan = samples[samples["source"].isin(["librivox", "music", "noise"])]
folder = musan["audio_path"].map(lambda path: Path(path).parent.name)
stratum = musan["locale"].where(musan["source"] == "librivox", folder).fillna("—")
for rows in musan.groupby([musan["source"], stratum]).groups.values():
    split[rows] = rows.map(assign(rows, MUSAN_TRAIN_SHARE))

is_target = samples["locale"].isin(TARGET_LOCALES) & samples["source"].isin(["commonvoice", "librivox"])
dataset = samples.assign(split=split, label=samples["locale"].where(is_target, "unknown"))
dataset = dataset[dataset["split"].notna()]

order = rng.permutation(len(dataset))
rank = dataset.iloc[order].groupby(["split", "locale", "speaker_id"]).cumcount().reindex(dataset.index)
cap = dataset["split"].map(MAX_CLIPS_PER_SPEAKER)
dataset = dataset[~((dataset["source"] == "commonvoice") & cap.notna() & (rank >= cap))].reset_index(drop=True)

print(f"Без выборки: {int(cv.sum() - known.sum())} записей других языков Common Voice, "
      f"{int((known & samples['speaker_id'].isna()).sum())} без диктора, "
      f"{int(leaked.sum())} et/fi от дикторов из train")
pd.crosstab([dataset["label"], dataset["source"], dataset["locale"].fillna("—")], dataset["split"],
            margins=True, margins_name="всего")[["train", "dev", "test", "всего"]]

Без выборки: 0 записей других языков Common Voice, 0 без диктора, 866 et/fi от дикторов из train


split                       train   dev  test  всего
label   source      locale                          
ar      commonvoice ar      11030   219   216  11465
        librivox    ar          0     1     0      1
ru      commonvoice ru      33583   726   729  35038
        librivox    ru          3     3     3      9
unknown commonvoice el       4191     0     0   4191
                    et          0  2518  2520   5038
                    fi          0   488   478    966
                    id       5350     0     0   5350
        librivox    da          0     0     1      1
                    de          5     6     5     16
                    en         23    27    26     76
                    es          3     4     4     11
                    fr          3     4     4     11
                    he          1     2     1      4
                    hu          0     0     1      1
                    it          2     3     3      8
                    ja          1     1     1      3
                    la          0     0     1      1
                    nl          3     3     3      9
                    pl          1     1     0      2
                    pt          3     3     3      9
                    tl          1     1     1      3
                    zh          2     2     3      7
                    —           0     0     1      1
        music       —         197   233   230    660
        noise       —         279   326   325    930
всего                       54681  4571  4559  63811

In [5]:
dataset.to_pickle("/kaggle/working/audiolid_split.pkl")

In [6]:
dataset.head(10)

,locale,speaker_id,audio_path,source,duration,split,label
0,ar,036b324cdc16640e1e7af9ad45f95b2f07fd5f6d1098b4...,/kaggle/input/datasets/manofwar/commonvoice-ar...,commonvoice,7.596,train,ar
1,ar,053a91dfd8c87629424edbb1a98cebea127137e6532174...,/kaggle/input/datasets/manofwar/commonvoice-ar...,commonvoice,3.936,train,ar
2,ar,073c6a4a1887384a1d6145a3258b7ca48de9f441c1762f...,/kaggle/input/datasets/manofwar/commonvoice-ar...,commonvoice,3.744,test,ar
3,ar,08ea1e4883b0112f794e6cba73552d702ba257721b88df...,/kaggle/input/datasets/manofwar/commonvoice-ar...,commonvoice,4.788,test,ar
4,ar,090215bbaa191ff831ef140914f9dd267f0a61aa5536eb...,/kaggle/input/datasets/manofwar/commonvoice-ar...,commonvoice,4.896,train,ar
5,ar,0aab971eac3eabea7027b19bedcde9f9bf1162f536326d...,/kaggle/input/datasets/manofwar/commonvoice-ar...,commonvoice,3.636,train,ar
6,ar,0ad95061b41fe74e207aa5b478970486358c8c40b4a6a8...,/kaggle/input/datasets/manofwar/commonvoice-ar...,commonvoice,8.280,train,ar
7,ar,0c8851d8362232c9d7b8d9f876d338781802fa2a7c06cb...,/kaggle/input/datasets/manofwar/commonvoice-ar...,commonvoice,4.572,train,ar
8,ar,0cae0344c4bc6379e9109c5925bc9f94a5e8b17a431058...,/kaggle/input/datasets/manofwar/commonvoice-ar...,commonvoice,3.336,train,ar
9,ar,0e01a07d994fb62d88792ff0bb4be7f51addbe483d1d62...,/kaggle/input/datasets/manofwar/commonvoice-ar...,commonvoice,4.608,train,ar


In [5]:
"""Посчитать признаки ECAPA для головы out и сохранить их на диск.

Признаки те же, что в ecapa_train.ipynb: центр записи до `seconds` секунд, моно,
8 кГц → 16 кГц, эмбеддинг ECAPA → act → norm → DNN классификатора, 512 чисел.
Вход — таблица выборок из audiolid.ipynb (audiolid_split.pkl) с колонками
audio_path, split, label; подойдут также .jsonl и .parquet с теми же колонками.

Kaggle::

    !python extract_ecapa_features.py --split-file /kaggle/working/audiolid_split.pkl

Результат — output/features_<seconds>s.pt со словарём:
    paths, split, label — списки в порядке строк таблицы;
    feats — тензор [N, 512], NaN в строках, которые не удалось прочитать;
    failed — {audio_path: причина}; meta — модель, ревизия, seconds.
Считается частями по --chunk-size записей; после обрыва повторный запуск
досчитывает только недостающие части.
"""

from __future__ import annotations

import argparse
import os
import sys
from itertools import groupby
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import soundfile as sf
import torch
from scipy.signal import resample_poly
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

MODEL_ID = "speechbrain/lang-id-voxlingua107-ecapa"
REVISION = "0253049ae131d6a4be1c4f0d8b0ff483a0f8c8e9"
FEATURE_DIM = 512


def load_clip(path, seconds):
    """Центр записи до `seconds` секунд, моно, 8 кГц → 16 кГц (как predict в ноутбуке)."""
    with sf.SoundFile(path) as audio:
        rate = audio.samplerate
        frames = int(seconds * rate)
        if audio.format == "MP3":
            # У MP3 длина в заголовке приблизительная. Берём центр декодированного клипа.
            samples = audio.read(dtype="float32", always_2d=True)
            offset = max(0, (len(samples) - frames) // 2)
            samples = samples[offset:offset + frames]
        else:
            audio.seek(max(0, (len(audio) - frames) // 2))
            samples = audio.read(frames, dtype="float32", always_2d=True)
    phone = resample_poly(samples.mean(axis=1), 8000, rate)
    return resample_poly(phone, 2, 1).astype(np.float32), len(samples) / rate


class Clips(Dataset):
    """Декодирует записи в процессах DataLoader; битый файл не останавливает прогон.

    Отдаёт массивы numpy, а не тензоры: воркерам не нужна разделяемая память torch,
    которой в контейнерах Kaggle бывает мало.
    """

    def __init__(self, paths, seconds):
        self.paths, self.seconds = list(paths), seconds

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            waveform, duration = load_clip(self.paths[i], self.seconds)
        except Exception as exc:
            return i, None, repr(exc)
        if duration < 0.05:  # ECAPA нужно несколько кадров звука
            return i, None, "короче 0.05 с"
        return i, waveform, None


def load_model(source, revision, device):
    from speechbrain.inference.classifiers import EncoderClassifier
    from speechbrain.utils.fetching import FetchConfig

    if Path(source).is_dir():  # локальная папка с hyperparams.yaml и чекпойнтами
        local = str(Path(source).resolve())
        return EncoderClassifier.from_hparams(
            source=local, overrides={"pretrained_path": local}, run_opts={"device": device},
            fetch_config=FetchConfig(allow_network=False),
        ).eval()
    return EncoderClassifier.from_hparams(
        source=source, run_opts={"device": device}, fetch_config=FetchConfig(revision=revision),
    ).eval()


@torch.no_grad()
def head_inputs(model, waveforms, device):
    """512 признаков на входе головы out для клипов одной длины."""
    clf = model.mods.classifier
    emb = model.encode_batch(torch.from_numpy(np.stack(waveforms)).to(device))  # [B, 1, 256]
    return clf.DNN(clf.norm(clf.act(emb)))[:, 0].float().cpu()                # [B, 512]


def extract_chunk(model, paths, seconds, device, batch_size, workers, progress):
    feats = torch.full((len(paths), FEATURE_DIM), float("nan"))
    failed = {}
    context = ("fork" if sys.platform.startswith("linux") else "spawn") if workers else None
    loader = DataLoader(Clips(paths, seconds), batch_size=batch_size, num_workers=workers,
                        collate_fn=list, multiprocessing_context=context)
    for batch in loader:
        good = []
        for i, waveform, error in batch:
            if error is None:
                good.append((i, waveform))
            else:
                failed[paths[i]] = error
        # Без паддинга: вместе считаются только клипы одной длины, поэтому признаки
        # совпадают с поклиповым расчётом. Почти все клипы ровно по `seconds` секунд.
        good.sort(key=lambda item: len(item[1]))
        for _, group in groupby(good, key=lambda item: len(item[1])):
            idx, waveforms = zip(*group)
            feats[list(idx)] = head_inputs(model, list(waveforms), device)
        progress.update(len(batch))
    return feats, failed


def read_table(path):
    path = Path(path)
    if path.suffix == ".pkl":
        table = pd.read_pickle(path)
    elif path.suffix == ".parquet":
        table = pd.read_parquet(path)
    elif path.suffix in (".jsonl", ".json"):
        table = pd.read_json(path, lines=True)
    else:
        raise ValueError(f"Неизвестный формат таблицы: {path}")
    missing = {"audio_path", "split", "label"} - set(table.columns)
    if missing:
        raise ValueError(f"В {path} нет колонок {sorted(missing)}")
    if table["audio_path"].isna().any():
        raise ValueError(f"В {path} есть строки без audio_path")
    return table.reset_index(drop=True)

In [ ]:
def extract_features_to_disk(
    split_file="/kaggle/working/audiolid_split.pkl",
    output="/kaggle/working/ecapa_features",
    seconds=3,
    model=MODEL_ID,               
    revision=REVISION,
    device="cuda:0" if torch.cuda.is_available() else "cpu",
    batch_size=None,           
    workers=os.cpu_count() or 0,  
    chunk_size=4096,             
):
    """Посчитать признаки для таблицы split_file и сохранить в output/features_<seconds>s.pt."""
    output = Path(output)
    batch_size = batch_size or (64 if device.startswith("cuda") else 8)
    seconds = int(seconds) if float(seconds).is_integer() else seconds
    table = read_table(split_file)
    paths = table["audio_path"].astype(str).tolist()
    parts_dir = output / f"parts_{seconds}s"
    result = output / f"features_{seconds}s.pt"
    meta = {"model": model, "revision": revision, "seconds": seconds, "feature": "classifier DNN output"}
    if result.exists():
        done = torch.load(result)
        if done["paths"] == paths and done["meta"] == meta:
            print(f"{result} уже посчитан для этой таблицы: {tuple(done['feats'].shape)}")
            return result

    parts_dir.mkdir(parents=True, exist_ok=True)
    chunks = [(start, paths[start:start + chunk_size]) for start in range(0, len(paths), chunk_size)]
    todo = []
    for start, chunk in chunks:
        part = parts_dir / f"{start:08d}.pt"
        if part.exists() and torch.load(part)["paths"] == chunk:
            continue
        todo.append((start, chunk, part))
    print(f"{len(paths)} записей, устройство {device}, батч {batch_size}, воркеров {workers}", flush=True)

    if todo:
        ecapa = load_model(model, revision, device)
        start_time = perf_counter()
        with tqdm(total=sum(len(chunk) for _, chunk, _ in todo), unit="клип", desc="Признаки") as progress:
            for start, chunk, part in todo:
                feats, failed = extract_chunk(ecapa, chunk, seconds, device, batch_size, workers, progress)
                torch.save({"paths": chunk, "feats": feats, "failed": failed}, part.with_suffix(".tmp"))
                part.with_suffix(".tmp").replace(part)  # часть появляется только целиком
        print(f"Посчитано за {perf_counter() - start_time:.0f} с", flush=True)

    feats, failed = [], {}
    for start, chunk in chunks:
        part = torch.load(parts_dir / f"{start:08d}.pt")
        feats.append(part["feats"])
        failed.update(part["failed"])
    saved = {"paths": paths, "feats": torch.cat(feats), "split": table["split"].astype(str).tolist(),
             "label": table["label"].astype(str).tolist(), "failed": failed, "meta": meta}
    torch.save(saved, result.with_suffix(".tmp"))
    result.with_suffix(".tmp").replace(result)
    for part in parts_dir.iterdir():
        part.unlink()
    parts_dir.rmdir()

    print(f"{result}: признаки {tuple(saved['feats'].shape)}, без признаков {len(failed)}")
    for path, reason in list(failed.items())[:5]:
        print(f"  {path}: {reason}")
    return result


features_file = extract_features_to_disk()

/kaggle/working/ecapa_features/features_3s.pt уже посчитан для этой таблицы: (63811, 512)
